In [78]:
import polars as pl
import polars.selectors as cs
from typing import Any, Tuple
import time
import jax.numpy as jnp
import jax
import jax_dataloader as jdl
from flax import nnx as nnx
import optax
import wandb
from flax.training import train_state
import orbax.checkpoint as ocp
from sklearn.model_selection import KFold
import numpy as np

In [33]:
cat_columns = {
    "MSSubClass": pl.Enum(['60', '20', '70', '50', '190', '90', '45', '120', '30', '85', '80', '160', '75', '180', '40', '150']),
    "MSZoning": pl.Enum(['RL', 'RM', 'C (all)', 'FV', 'RH']),
    "Street": pl.Enum(['Pave', 'Grvl']),
    "Alley": pl.Enum(['Grvl', 'Pave']),
    "LotShape": pl.Enum(['Reg', 'IR1', 'IR2', 'IR3']),
    "LandContour": pl.Enum(['Lvl', 'Bnk', 'HLS', 'Low']),
    "Utilities": pl.Enum(['AllPub', 'NoSeWa']),
    "LotConfig": pl.Enum(['Inside', 'FR2', 'Corner', 'CulDSac', 'FR3']),
    "LandSlope": pl.Enum(['Gtl', 'Mod', 'Sev']),
    "Neighborhood": pl.Enum(['CollgCr', 'Veenker', 'Crawfor', 'NoRidge', 'Mitchel', 'Somerst', 'NWAmes', 'OldTown', 'BrkSide', 'Sawyer', 'NridgHt', 'IDOTRR', 'MeadowV', 'NAmes', 'SawyerW', 'Edwards', 'Timber', 'Gilbert', 'StoneBr', 'ClearCr', 'NPkVill', 'Blmngtn', 'BrDale', 'SWISU', 'Blueste']),
    "Condition1": pl.Enum(['Norm', 'Feedr', 'PosN', 'Artery', 'RRAe', 'RRAn', 'RRNn', 'PosA', 'RRNe']),
    "Condition2": pl.Enum(['Norm', 'Artery', 'RRNn', 'Feedr', 'PosN', 'PosA', 'RRAn', 'RRAe']),
    "BldgType": pl.Enum(['1Fam', '2fmCon', 'TwnhsE', 'Duplex', 'Twnhs']),
    "HouseStyle": pl.Enum(['2Story', '1Story', '1.5Fin', '1.5Unf', 'SFoyer', 'SLvl', '2.5Unf', '2.5Fin']),
    "OverallQual": pl.Enum(['7', '6', '8', '5', '9', '4', '10', '3', '1', '2']),
    "OverallCond": pl.Enum(['5', '8', '6', '7', '4', '2', '3', '9', '1']),
    "RoofStyle": pl.Enum(['Gable', 'Hip', 'Gambrel', 'Mansard', 'Flat', 'Shed']),
    "RoofMatl": pl.Enum(['CompShg', 'WdShngl', 'Metal', 'WdShake', 'Membran', 'Tar&Grv', 'Roll', 'ClyTile']),
    "Exterior1st": pl.Enum(['VinylSd', 'MetalSd', 'Wd Sdng', 'HdBoard', 'BrkFace', 'WdShing', 'CemntBd', 'Plywood', 'AsbShng', 'Stucco', 'BrkComm', 'AsphShn', 'Stone', 'ImStucc', 'CBlock']),
    "Exterior2nd": pl.Enum(['VinylSd', 'MetalSd', 'Wd Shng', 'HdBoard', 'CmentBd', 'Plywood', 'Wd Sdng', 'BrkFace', 'Stucco', 'AsbShng', 'Brk Cmn', 'ImStucc', 'AsphShn', 'Stone', 'Other', 'CBlock']),
    "MasVnrType": pl.Enum(['BrkFace', 'None', 'Stone', 'BrkCmn']),
    "ExterQual": pl.Enum(['Gd', 'TA', 'Ex', 'Fa']),
    "ExterCond": pl.Enum(['TA', 'Gd', 'Fa', 'Po', 'Ex']),
    "Foundation": pl.Enum(['PConc', 'CBlock', 'BrkTil', 'Wood', 'Slab', 'Stone']),
    "BsmtQual": pl.Enum(['Gd', 'TA', 'Ex', 'Fa']),
    "BsmtCond": pl.Enum(['TA', 'Gd', 'Fa', 'Po']),
    "BsmtExposure": pl.Enum(['No', 'Gd', 'Mn', 'Av']),
    "BsmtFinType1": pl.Enum(['GLQ', 'ALQ', 'Unf', 'Rec', 'LwQ', 'BLQ']),
    "BsmtFinType2": pl.Enum(['Unf', 'BLQ', 'ALQ', 'Rec', 'LwQ', 'GLQ']),
    "Heating": pl.Enum(['GasA', 'GasW', 'Grav', 'Wall', 'OthW', 'Floor']),
    "HeatingQC": pl.Enum(['Ex', 'Gd', 'TA', 'Fa', 'Po']),
    "CentralAir": pl.Enum(['Y', 'N']),
    "Electrical": pl.Enum(['SBrkr', 'FuseF', 'FuseA', 'FuseP', 'Mix']),
    "KitchenQual": pl.Enum(['Gd', 'TA', 'Ex', 'Fa']),
    "Functional": pl.Enum(['Typ', 'Min1', 'Maj1', 'Min2', 'Mod', 'Maj2', 'Sev']),
    "FireplaceQu": pl.Enum(['TA', 'Gd', 'Fa', 'Ex', 'Po']),
    "GarageType": pl.Enum(['Attchd', 'Detchd', 'BuiltIn', 'CarPort', 'Basment', '2Types']),
    "GarageFinish": pl.Enum(['RFn', 'Unf', 'Fin']),
    "GarageQual": pl.Enum(['TA', 'Fa', 'Gd', 'Ex', 'Po']),
    "GarageCond": pl.Enum(['TA', 'Fa', 'Gd', 'Po', 'Ex']),
    "PavedDrive": pl.Enum(['Y', 'N', 'P']),
    "PoolQC": pl.Enum(['Ex', 'Fa', 'Gd']),
    "Fence": pl.Enum(['MnPrv', 'GdWo', 'GdPrv', 'MnWw']),
    "MiscFeature": pl.Enum(['Shed', 'Gar2', 'Othr', 'TenC']),
    "SaleType": pl.Enum(['WD', 'New', 'COD', 'ConLD', 'ConLI', 'CWD', 'ConLw', 'Con', 'Oth']),
    "SaleCondition": pl.Enum(['Normal', 'Abnorml', 'Partial', 'AdjLand', 'Alloca', 'Family']),
}

In [65]:
raw_train_df = pl.read_csv("./train.csv",
                     null_values="NA",
                     schema_overrides=cat_columns)

raw_test_df = pl.read_csv("./test.csv",
                     null_values="NA",
                     schema_overrides=cat_columns).with_columns(pl.lit(None, dtype=pl.Int64).alias("SalePrice"))

print(raw_train_df.shape, raw_test_df.shape)
# polars one hot encoding does not use the values defined in the enums but dynamically based on the current values in the dataframe.
# Therefore, in order to get identical encodes, we run to_dummies on the entire dataframe.
full_df = pl.concat([raw_train_df, raw_test_df]).to_dummies(cs.by_dtype(pl.Enum))

raw_train_df = full_df[0:raw_train_df.shape[0]]
raw_test_df = full_df[raw_train_df.shape[0]:]

raw_train_df.shape, raw_test_df.shape

# df_cat = raw_train_df.select(cs.categorical())

# # 3) Build up a dict of name → list of categories
# enum_map: dict[str, list[str]] = {}
# for col in df_cat.columns:
#     cats = df_cat[col].cat.get_categories()
#     enum_map[col] = cats.to_list()

# # 4) Print out the Python dict you can drop into your code
# print("cat_columns = {")
# for name, cats in enum_map.items():
#     # repr(cats) will give you a valid Python list literal
#     print(f'    "{name}": pl.Enum({repr(cats)}),' )
# print("}")

(1460, 81) (1459, 81)


((1460, 345), (1459, 345))

In [60]:
def process_df(df: pl.DataFrame, mean: pl.DataFrame|None=None, std: pl.DataFrame|None=None):
    df = df.drop("Id")

    df = df.with_columns(
        (pl.col("GarageYrBlt").is_null().alias("HasGarage")),
        (pl.col("YrSold") - pl.col("GarageYrBlt"))
        .alias("GarageYrBlt"),
        (pl.col("YrSold") - pl.col("YearBuilt"))
        .alias("Age"),
        (pl.col("TotalBsmtSF") + pl.col("GrLivArea"))
        .alias("TotalSF"),
        pl.col("SalePrice").log()
    )

    if mean is None: mean = df.mean()
    if std is None: std = df.std()

    df = df.with_columns(
        [(pl.col(c) - mean[c])/std[c] for c in df.select((cs.numeric() - (cs.by_dtype(pl.UInt8))).exclude("SalePrice")).columns]
    )

    df = df.fill_null(0) # We fill LotFrontage and MasVnrArea NA with 0

    df = df.select(
        pl.all().exclude("SalePrice"),
        pl.col("SalePrice")
    )

    return (df, (mean, std))

train_df, (_, _) = process_df(raw_train_df)
train_df.shape

(1460, 347)

In [57]:
def loss_fn(model, batch):
    x, y = batch
    y_pred = model(x).squeeze(-1)
    # jax.debug.print("y {}", y)
    # print(y_pred.shape, y.shape)
    assert y_pred.shape == y.shape
    return jnp.sqrt(jnp.mean((y_pred - y) ** 2))

@nnx.jit
def train_step(model: nnx.Module, optimiser: nnx.Optimizer, metrics: nnx.MultiMetric, batch: Tuple[jax.Array, jax.Array]):
    model.train()
    loss, grads = nnx.value_and_grad(loss_fn)(model, batch)
    metrics.update(loss=loss)
    optimiser.update(grads)

@nnx.jit
def eval_step(model: nnx.Module, metrics: nnx.MultiMetric, batch: tuple[jax.Array, jax.Array]):
    model.eval()
    loss = loss_fn(model, batch)
    metrics.update(loss=loss)

In [6]:
# Using linear net for now

# class MLP(nnx.Module):
#     def __init__(self, din, dhidden, dout, rngs: nnx.Rngs):
#         self.linear1 = nnx.Linear(din, dhidden, rngs=rngs)
#         self.drop1 = nnx.Dropout(0.3, rngs=rngs)
#         self.linear2 = nnx.Linear(dhidden, dhidden, rngs=rngs)
#         self.drop2 = nnx.Dropout(0.5, rngs=rngs)
#         self.linear3 = nnx.Linear(dhidden, dout, rngs=rngs)

#     def __call__(self, X):
#         H1 = nnx.relu(self.linear1(X))
#         H1 = self.drop1(H1)
#         H2 = nnx.relu(self.linear2(H1))
#         H2 = self.drop2(H2)
#         return self.linear3(H2)


In [61]:
def train_model(epochs, dl_train:jdl.DataLoader, dl_val:jdl.DataLoader|None = None):
    rngs = nnx.Rngs(default=0)
    # model = MLP(336, 512, 1, rngs=rngs)
    model = nnx.Linear(346, 1, rngs=rngs)
    optimiser = nnx.Optimizer(model, optax.sgd(1e-2))
    metrics = nnx.MultiMetric(
        loss = nnx.metrics.Average("loss")
    )

    final_train_loss = 0
    final_val_loss = 0

    for i in range(epochs):
        for batch in dl_train:
            train_step(model, optimiser, metrics, batch)

        m = metrics.compute()
        metrics.reset()
        final_train_loss = m["loss"].item()

        # print(f"epoch {i+1} of {epochs} with loss = {m["loss"]}")

        if (dl_val is not None):
            for batch in dl_val:
                eval_step(model, metrics, batch)

            m = metrics.compute()
            metrics.reset()
            final_val_loss = m["loss"].item()
        # print(f"epoch {i+1} of {epochs} with loss = {m["loss"]}")

    return (final_train_loss, final_val_loss), model
        

In [62]:
kf = KFold(n_splits=5, shuffle=True)

# arr = train_df.to_jax()
losses = jnp.array([])

for i, (train_index, test_index) in enumerate(kf.split(raw_train_df)): # type: ignore

    train_df, (mean, std) = process_df(raw_train_df[train_index])
    val_df, (_, _) = process_df(raw_train_df[test_index], mean, std)

    # print(val_df.shape)

    train_arr = train_df.to_jax()
    val_arr = val_df.to_jax()

    X_train, y_train = train_arr[:, :-1], train_arr[:, -1]
    X_val,   y_val   = val_arr[:,  :-1], val_arr[:,  -1]

    # print(y_train)

    train_ds = jdl.ArrayDataset(X_train, y_train)
    val_ds   = jdl.ArrayDataset(X_val,   y_val)

    dl_train = jdl.DataLoader(train_ds, 'jax',
                              batch_size=64, shuffle=True)
    dl_val   = jdl.DataLoader(val_ds,   'jax',
                              batch_size=64, shuffle=False)
    
    (_, val_loss), _ = train_model(100, dl_train, dl_val)
    losses = jnp.append(losses, val_loss)

    print(f"fold {i} loss = {val_loss}")

# print(f"final loss = {losses.mean()}")

fold 0 loss = 0.20783951878547668
fold 1 loss = 0.1661377102136612
fold 2 loss = 0.17086446285247803
fold 3 loss = 0.17319545149803162
fold 4 loss = 0.21813717484474182


In [63]:
# Train a model on the entire dataset

train_entire_df, (mean, std) = process_df(raw_train_df)
train_entire_array = train_entire_df.to_jax()

X_train, y_train = train_entire_array[:, :-1], train_entire_array[:, -1]

train_ds = jdl.ArrayDataset(X_train, y_train)
train_dl = jdl.DataLoader(train_ds, 'jax',
                              batch_size=64, shuffle=True)

loss, model = train_model(100, train_dl)
loss

(0.13596105575561523, 0)

In [87]:



test_df, (_, _) = process_df(raw_test_df, mean, std)

test_array = test_df.to_jax()

X_test = test_array[:, :-1]

pred = model(X_test)

final = pl.concat([raw_test_df.select(pl.col("Id")), pl.from_numpy(np.array(pred), schema=["SalePrice"])], how="horizontal")

final = final.with_columns(
    pl.col("SalePrice").exp()
)

final.write_csv("predictions.csv")